# D1.7 · Drift monitoring

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.6 · Distinguishing agent from human](https://spbreed.github.io/cyber-commons/lessons/D1.6.html)**.

| | |
|---|---|
| Tools used | promptfoo, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Change the model underneath and catch the detection regression.

**Why a security engineer needs it.** A detection that worked last month is silently degraded. The control it builds is: watch model updates, prompt changes, index refreshes, tool versions.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Nothing was attacked. The model was upgraded, a prompt was edited, a tool changed its output format — and the behaviour of the system moved. Drift is the failure mode with no adversary, and it is far more common than the ones with one.

> **At CyberTravels.** Nothing was attacked. The model provider upgraded, Alex edited a prompt, the tool manifest changed — and CyberTravels' baseline moved underneath every detection built on it.

## 2 · The framework

```
   nothing was attacked

   model upgraded ----+
   prompt edited  ----+---> behaviour moves ---> baselines stale
   tool changed   ----+                          detections silent

   drift is the failure mode with no adversary, and the common one
   the control: a fixed probe suite, run on every change
```

Drift monitoring exists because an agent's behaviour changes **without a code
change**. A new model version, an edited prompt, an added tool — none of these
pass through the change management process built for code, and all of them
invalidate the testing your controls were signed off against.

That is the precise claim: the control was tested against a behaviour that no
longer exists. It has not failed; it is *unevidenced*, which is a different and
more honest state.

Two things are needed:

1. A **signed-off baseline** — what normal looked like when the control passed.
2. A **freshness window** on the control test, derived from how fast the thing
   it tests actually drifts.

E1.7 turns the second into a compliance posture. This lesson produces the signal.

## 3 · Demo — drift across a quarter

In [ ]:
import time
from dataclasses import dataclass, field

now = time.time(); DAY = 86400

@dataclass
class Baseline:
    signed_off: float
    tool_mix: dict
    def compare(self, mix):
        total = sum(mix.values()) or 1
        cur = {k: v/total for k, v in mix.items()}
        keys = set(cur) | set(self.tool_mix)
        tvd = sum(abs(cur.get(k,0) - self.tool_mix.get(k,0)) for k in keys)/2
        return {"drift": round(tvd, 3),
                "new_tools": sorted(set(cur) - set(self.tool_mix)),
                "gone": sorted(set(self.tool_mix) - set(cur))}

base = Baseline(signed_off=now - 90*DAY,
                tool_mix={"read_file": 0.80, "search": 0.15, "write_file": 0.05})

TIMELINE = [
 (now - 90*DAY, "control signed off",     {"read_file": 800, "search": 150, "write_file": 50}),
 (now - 60*DAY, "prompt edited",          {"read_file": 700, "search": 150, "write_file": 150}),
 (now - 30*DAY, "tool added (no PR)",     {"read_file": 500, "search": 120, "write_file": 180,
                                           "run_shell": 200}),
 (now -  5*DAY, "model upgraded by vendor",{"read_file": 300, "search": 100, "write_file": 250,
                                            "run_shell": 350}),
]
print(f"{'when':>8}  {'event':26s}{'drift':>7}  new tools")
print("-" * 66)
for ts, event, mix in TIMELINE:
    d = base.compare(mix)
    print(f"{(now-ts)/DAY:>6.0f}d  {event:26s}{d['drift']:>7.3f}  {d['new_tools']}")

## 4 · Where it breaks — none of these was a code change

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">change surface</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">in change management?</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what happens today</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">application code</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">pull request, review, CI</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agent prompt</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">edited in a console</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">tool manifest</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a config change, with no threat-model diff</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">model version</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">provider-side; you may not be told</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">policy</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">if it is in git — often it is not</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">approval settings</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a toggle in an admin UI</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Four of six surfaces bypass change management entirely. Drift is the failure mode with no adversary, and this table is why it is also the failure mode with no ticket.</div>

## 5 · The control — freshness derived from the observed drift rate

In [ ]:
def drift_rate(baseline, timeline):
    """How fast does this agent actually drift? Set the window from the answer."""
    pts = [(ts, baseline.compare(mix)["drift"]) for ts, _, mix in timeline]
    pts.sort()
    span_days = (pts[-1][0] - pts[0][0]) / 86400
    return (pts[-1][1] - pts[0][1]) / max(span_days, 1)

rate = drift_rate(base, TIMELINE)
TOLERANCE = 0.25
window = int(TOLERANCE / rate) if rate > 0 else 365
print(f"observed drift rate  {rate:.5f} TVD/day")
print(f"tolerance            {TOLERANCE}")
print(f"→ freshness window   {window} days "
      f"(a control test older than this is unevidenced, not passing)")

@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        age = (at - self.tested_at) / 86400
        if age > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

tests = [ControlTest("SB-1", True, now - 90*DAY, window),
         ControlTest("SB-2", True, now - 10*DAY, window),
         ControlTest("DR-1", False, now, window)]
print(f"\n{'control':10s}{'age (d)':>9}{'state':>10}")
print("-" * 30)
for t in tests:
    print(f"{t.cid:10s}{(now-t.tested_at)/DAY:>9.0f}{t.state(now):>10}")
evidenced = sum(t.state(now) == "PASS" for t in tests)
print(f"\ncurrently evidenced: {evidenced}/{len(tests)}")
assert any(t.state(now) == "STALE" for t in tests)

## What you just proved

Drift rises across the quarter from 0.0 at sign-off to roughly 0.35 after the model upgrade, with `run_shell` appearing as a new tool. Four of six change surfaces bypass change management. The observed drift rate yields a freshness window, and the 90-day-old control test is reported STALE rather than passing.

## Your turn

Compute the drift rate for one production agent from three months of telemetry, and set its control freshness window from that number rather than from the audit calendar.

---

**Next → [D1.8 · Threat intel sub-lane](https://spbreed.github.io/cyber-commons/lessons/D1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*